<a href="https://colab.research.google.com/github/Saiji/Data-Science-Work/blob/master/Demand_Forecasting_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Forecasting Libraries
from prophet import Prophet
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")

# ==========================================
# 1. DATA LOADING & PREPROCESSING
# ==========================================
def load_and_preprocess_data(file_path: str, date_col: str = "date", demand_col: str = "demand") -> pd.DataFrame:
    """
    Loads raw sales data, parses dates, and aggregates to monthly frequency (Month Start).
    """
    path = Path(file_path)
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_csv(file_path)

    # Ensure correct column naming
    if date_col not in df.columns or demand_col not in df.columns:
        raise ValueError(f"CSV must contain '{date_col}' and '{demand_col}' columns.")

    # Convert dates and sort
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(by=date_col)

    # Aggregate daily/transactional data to Monthly Start ('MS') demand
    monthly_df = (
        df.set_index(date_col)[demand_col]
        .resample("MS")
        .sum()
        .reset_index()
    )

    # Filter out partial leading/trailing zero months if needed
    monthly_df[demand_col] = monthly_df[demand_col].clip(lower=0)

    return monthly_df


# ==========================================
# 2. EVALUATION METRICS
# ==========================================
def calculate_wape(actual: np.ndarray, forecast: np.ndarray) -> float:
    """Calculates Weighted Absolute Percentage Error (WAPE). Preferred over MAPE for inventory."""
    return (np.sum(np.abs(actual - forecast)) / np.sum(actual)) * 100


# ==========================================
# 3. FORECAST ENGINES
# ==========================================
def forecast_prophet(df: pd.DataFrame, forecast_horizon: int = 12) -> pd.DataFrame:
    """Generates monthly demand forecast using Meta Prophet."""
    # Prepare Prophet dataframe format
    prophet_df = df.rename(columns={"date": "ds", "demand": "y"})

    # Configure Prophet for monthly demand
    model = Prophet(
        growth="linear",
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive",  # Change to 'multiplicative' if demand variance scales with volume
        interval_width=0.80           # 80% confidence interval for safety stock planning
    )
    model.fit(prophet_df)

    # Generate future dates
    future = model.make_future_dataframe(periods=forecast_horizon, freq="MS")
    forecast = model.predict(future)

    # Standardize output structure
    output = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].rename(
        columns={
            "ds": "date",
            "yhat": "forecast_demand",
            "yhat_lower": "lower_bound",
            "yhat_upper": "upper_bound"
        }
    )
    # Clip negative values
    output["forecast_demand"] = output["forecast_demand"].clip(lower=0)
    output["lower_bound"] = output["lower_bound"].clip(lower=0)

    return output


def forecast_statsmodels(df: pd.DataFrame, forecast_horizon: int = 12) -> pd.DataFrame:
    """Fallback engine using Holt-Winters Exponential Smoothing via Statsmodels."""
    ts = df.set_index("date")["demand"].asfreq("MS")

    # Fit Holt-Winters Exponential Smoothing model
    model = ExponentialSmoothing(
        ts,
        trend="add",
        seasonal="add",
        seasonal_periods=12,
        initialization_method="estimated"
    ).fit()

    future_dates = pd.date_range(start=ts.index[-1] + pd.DateOffset(months=1), periods=forecast_horizon, freq="MS")
    forecast_vals = model.forecast(forecast_horizon)

    output = pd.DataFrame({
        "date": list(ts.index) + list(future_dates),
        "forecast_demand": list(model.fittedvalues) + list(forecast_vals),
        "lower_bound": np.nan,
        "upper_bound": np.nan
    })
    output["forecast_demand"] = output["forecast_demand"].clip(lower=0)
    return output


# ==========================================
# 4. MAIN PIPELINE EXECUTION
# ==========================================
def run_pipeline(
    csv_path: str,
    output_path: str = "monthly_demand_forecast.csv",
    forecast_horizon: int = 12,
    engine: str = "prophet"
):
    print(f"--- Processing File: {csv_path} ---")
    df = load_and_preprocess_data(csv_path)
    print(f"Loaded {len(df)} months of historical data from {df['date'].min().strftime('%Y-%m')} to {df['date'].max().strftime('%Y-%m')}.")

    # --- Backtesting Validation (Holdout last 6 months) ---
    if len(df) > 18:
        train_df = df.iloc[:-6]
        test_df = df.iloc[-6:]

        if engine == "prophet":
            val_results = forecast_prophet(train_df, forecast_horizon=6)
            preds = val_results.tail(6)["forecast_demand"].values
        else:
            val_results = forecast_statsmodels(train_df, forecast_horizon=6)
            preds = val_results.tail(6)["forecast_demand"].values

        wape = calculate_wape(test_df["demand"].values, preds)
        print(f"Out-of-sample Backtest Error (WAPE - Last 6 Months): {wape:.2f}%")
    else:
        print("Dataset too short for backtesting; skipping validation step.")

    # --- Full Out-of-Sample Forecast ---
    if engine == "prophet":
        forecast_df = forecast_prophet(df, forecast_horizon=forecast_horizon)
    else:
        forecast_df = forecast_statsmodels(df, forecast_horizon=forecast_horizon)

    # Merge actuals with forecasts
    final_df = pd.merge(df, forecast_df, on="date", how="right")
    final_df = final_df.rename(columns={"demand": "actual_demand"})

    # Export to CSV
    final_df.to_csv(output_path, index=False)
    print(f"\nForecast successfully saved to: {output_path}")

    # Display Next 6 Months Preview
    print("\nNext 6 Months Demand Forecast Preview:")
    print(final_df.tail(forecast_horizon)[["date", "forecast_demand", "lower_bound", "upper_bound"]].head(6).to_string(index=False))

    return final_df


if __name__ == "__main__":
    # Example usage:
    # CSV file requirements: Must have columns 'date' (YYYY-MM-DD) and 'demand' (numeric)

    # 1. Generate Dummy Data File for testing (Remove this section if running on real file)
    sample_dates = pd.date_range(start="2023-01-01", end="2026-06-01", freq="MS")
    sample_demand = np.round(100 + np.sin(np.arange(len(sample_dates))) * 20 + np.random.normal(0, 5, len(sample_dates)))
    pd.DataFrame({"date": sample_dates, "demand": sample_demand}).to_csv("sample_sales.csv", index=False)

    # 2. Run Demand Planning Engine
    forecast_results = run_pipeline(
        csv_path="sample_sales.csv",
        output_path="monthly_demand_forecast.csv",
        forecast_horizon=12,
        engine="prophet"  # Options: 'prophet' or 'statsmodels'
    )

--- Processing File: sample_sales.csv ---
Loaded 42 months of historical data from 2023-01 to 2026-06.
Out-of-sample Backtest Error (WAPE - Last 6 Months): 12.75%

Forecast successfully saved to: monthly_demand_forecast.csv

Next 6 Months Demand Forecast Preview:
      date  forecast_demand  lower_bound  upper_bound
2026-07-01        88.103550    79.292591    96.762843
2026-08-01       107.417061    98.722090   116.628499
2026-09-01       116.782804   108.082223   125.268102
2026-10-01       115.115758   106.364461   123.062693
2026-11-01        95.802902    86.652698   104.239880
2026-12-01        83.987686    74.898359    92.390962
